In [12]:
# === Cell 1: Imports & config (Notebook-safe) ===

import sys
import json
from pathlib import Path

import pymysql
import requests
import matplotlib.pyplot as plt
import pandas as pd

# Notebook-safe: allow importing prompts.py
SCRIPT_DIR = Path().resolve()
PROJECT_ROOT = SCRIPT_DIR
sys.path.insert(0, str(PROJECT_ROOT))

from prompts import (
    build_few_shot_prompt,
    build_cot_prompt,
    build_ltm_prompt,
    build_eg_prompt,
    build_refine_prompt,  
)

MYSQL_HOST = "localhost"
MYSQL_USER = "root"
MYSQL_PASSWORD = "test"
MYSQL_DB = "adultdb"
TABLE_NAME = "adult_income"

MODEL_ENDPOINT = "http://localhost:11437/v1/chat/completions"
MODEL_NAME = "arctic-finetuned"

PROMPTING_TECHNIQUES = ["few_shot", "cot", "ltm", "eg"]

print("Loaded notebook config OK.")


Loaded notebook config OK.


In [13]:
# === Cell 2: Test cases (100 questions) ===

# IMPORTANT:
# Copy the full TEST_CASES list from your ak_model_test.py and paste it here.
# It must be a list of dicts like:
#   { "question": "...", "gold_sql": "..." }

TEST_CASES = [
        {
            "question": "What is the total number of people in each workclass?",
            "gold_sql": "SELECT workclass, COUNT(*) AS total FROM adult_income GROUP BY workclass ORDER BY total DESC;"
        },
        {
            "question": "For each race, show the average age and average hours per week.",
            "gold_sql": "SELECT race, AVG(age) AS avg_age, AVG(hours_per_week) AS avg_hours FROM adult_income GROUP BY race ORDER BY avg_age DESC;"
        },
        {
            "question": "Which education level has the highest percentage of people earning more than 50K?",
            "gold_sql": "SELECT education, 100.0 * AVG(CASE WHEN income = '>50K' THEN 1.0 ELSE 0.0 END) AS pct_high_income FROM adult_income GROUP BY education ORDER BY pct_high_income DESC LIMIT 1;"
        },
        {
            "question": "What is the average capital gain for people who work more than 50 hours per week?",
            "gold_sql": "SELECT AVG(capital_gain) AS avg_capital_gain FROM adult_income WHERE hours_per_week > 50;"
        },
        {
            "question": "For each marital status, show the count and percentage earning more than 50K.",
            "gold_sql": "SELECT marital_status, COUNT(*) AS total, 100.0 * AVG(CASE WHEN income = '>50K' THEN 1.0 ELSE 0.0 END) AS pct_high_income FROM adult_income GROUP BY marital_status ORDER BY pct_high_income DESC;"
        },
        {
            "question": "Which occupation has the highest average capital gain (ignore '?', require at least 20 people)?",
            "gold_sql": "SELECT occupation, AVG(capital_gain) AS avg_capital_gain FROM adult_income WHERE occupation <> '?' GROUP BY occupation HAVING COUNT(*) >= 20 ORDER BY avg_capital_gain DESC LIMIT 1;"
        },
        {
            "question": "For each relationship type, what is the average age by sex?",
            "gold_sql": "SELECT relationship, sex, AVG(age) AS avg_age FROM adult_income GROUP BY relationship, sex ORDER BY relationship, sex;"
        },
        {
            "question": "What percentage of women earn more than 50K compared to men?",
            "gold_sql": "SELECT sex, 100.0 * AVG(CASE WHEN income = '>50K' THEN 1.0 ELSE 0.0 END) AS pct_high_income FROM adult_income GROUP BY sex ORDER BY sex;"
        },
        {
            "question": "For each native country with at least 50 people, show the average educational_num.",
            "gold_sql": "SELECT native_country, AVG(educational_num) AS avg_educational_num FROM adult_income GROUP BY native_country HAVING COUNT(*) >= 50 ORDER BY avg_educational_num DESC;"
        },
        {
            "question": "Which workclass has the highest average hours worked per week?",
            "gold_sql": "SELECT workclass, AVG(hours_per_week) AS avg_hours FROM adult_income GROUP BY workclass ORDER BY avg_hours DESC LIMIT 1;"
        },
        {
            "question": "For each education level, show the distribution by sex (count and percentage).",
            "gold_sql": "SELECT education, sex, COUNT(*) AS cnt, 100.0 * COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY education) AS pct FROM adult_income GROUP BY education, sex ORDER BY education, cnt DESC;"
        },
        {
            "question": "What is the average age of people earning more than 50K by occupation (top 10, ignore '?')?",
            "gold_sql": "SELECT occupation, AVG(age) AS avg_age FROM adult_income WHERE income = '>50K' AND occupation <> '?' GROUP BY occupation ORDER BY avg_age DESC LIMIT 10;"
        },
        {
            "question": "For each race and sex combination, what is the average hours worked per week?",
            "gold_sql": "SELECT race, sex, AVG(hours_per_week) AS avg_hours FROM adult_income GROUP BY race, sex ORDER BY race, sex;"
        },
        {
            "question": "Which marital status has the highest percentage of people working more than 40 hours per week?",
            "gold_sql": "SELECT marital_status, 100.0 * AVG(CASE WHEN hours_per_week > 40 THEN 1.0 ELSE 0.0 END) AS pct_over_40hrs FROM adult_income GROUP BY marital_status ORDER BY pct_over_40hrs DESC LIMIT 1;"
        },
        {
            "question": "For each occupation, show the average capital loss (ignore '?', require at least 25 people).",
            "gold_sql": "SELECT occupation, AVG(capital_loss) AS avg_capital_loss FROM adult_income WHERE occupation <> '?' GROUP BY occupation HAVING COUNT(*) >= 25 ORDER BY avg_capital_loss DESC;"
        },
        {
            "question": "What is the average educational_num for people earning <=50K vs >50K?",
            "gold_sql": "SELECT income, AVG(educational_num) AS avg_educational_num FROM adult_income GROUP BY income ORDER BY income;"
        },
        {
            "question": "For each native country, show the top occupation by count (ignore '?', require at least 30 people in country).",
            "gold_sql": "WITH occ AS (SELECT native_country, occupation, COUNT(*) AS cnt FROM adult_income WHERE occupation <> '?' GROUP BY native_country, occupation), ranked AS (SELECT native_country, occupation, cnt, ROW_NUMBER() OVER (PARTITION BY native_country ORDER BY cnt DESC) AS rn FROM occ) SELECT r.native_country, r.occupation AS top_occupation, r.cnt FROM ranked r JOIN (SELECT native_country, COUNT(*) AS total FROM adult_income GROUP BY native_country HAVING total >= 30) c ON r.native_country = c.native_country WHERE r.rn = 1 ORDER BY r.cnt DESC;"
        },
        {
            "question": "Which relationship type has the highest average capital gain?",
            "gold_sql": "SELECT relationship, AVG(capital_gain) AS avg_capital_gain FROM adult_income GROUP BY relationship ORDER BY avg_capital_gain DESC LIMIT 1;"
        },
        {
            "question": "For each workclass and sex, what is the average hours worked per week?",
            "gold_sql": "SELECT workclass, sex, AVG(hours_per_week) AS avg_hours FROM adult_income GROUP BY workclass, sex ORDER BY workclass, sex;"
        },
        {
            "question": "What is the average age by income class and sex?",
            "gold_sql": "SELECT income, sex, AVG(age) AS avg_age FROM adult_income GROUP BY income, sex ORDER BY income, sex;"
        },
        {
            "question": "For each education level, show the percentage of people in each income bracket.",
            "gold_sql": "SELECT education, income, 100.0 * COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY education) AS pct FROM adult_income GROUP BY education, income ORDER BY education, income;"
        },
        {
            "question": "Which occupations have the highest percentage of people working exactly 40 hours per week (ignore '?', require at least 20 people)?",
            "gold_sql": "SELECT occupation, 100.0 * AVG(CASE WHEN hours_per_week = 40 THEN 1.0 ELSE 0.0 END) AS pct_40hrs, COUNT(*) AS cnt FROM adult_income WHERE occupation <> '?' GROUP BY occupation HAVING cnt >= 20 ORDER BY pct_40hrs DESC LIMIT 10;"
        },
        {
            "question": "For each race, what is the most common workclass and its count?",
            "gold_sql": "WITH wc AS (SELECT race, workclass, COUNT(*) AS cnt FROM adult_income GROUP BY race, workclass), ranked AS (SELECT race, workclass, cnt, ROW_NUMBER() OVER (PARTITION BY race ORDER BY cnt DESC, workclass ASC) AS rn FROM wc) SELECT race, workclass AS top_workclass, cnt FROM ranked WHERE rn = 1 ORDER BY cnt DESC;"
        },
        {
            "question": "What is the average hours per week for people with capital_gain > 0 vs capital_gain = 0?",
            "gold_sql": "SELECT CASE WHEN capital_gain > 0 THEN 'Has Gain' ELSE 'No Gain' END AS gain_status, AVG(hours_per_week) AS avg_hours FROM adult_income GROUP BY gain_status;"
        },
        {
            "question": "For each sex, show the top 5 education levels by count.",
            "gold_sql": "WITH edu AS (SELECT sex, education, COUNT(*) AS cnt FROM adult_income GROUP BY sex, education), ranked AS (SELECT sex, education, cnt, ROW_NUMBER() OVER (PARTITION BY sex ORDER BY cnt DESC, education ASC) AS rn FROM edu) SELECT sex, education, cnt FROM ranked WHERE rn <= 5 ORDER BY sex, cnt DESC;"
        },
        {
            "question": "Which native country has the highest average educational_num (require at least 40 people)?",
            "gold_sql": "SELECT native_country, AVG(educational_num) AS avg_educational_num FROM adult_income GROUP BY native_country HAVING COUNT(*) >= 40 ORDER BY avg_educational_num DESC LIMIT 1;"
        },
        {
            "question": "For each occupation, show the average age and average hours per week (ignore '?', top 15 by average age).",
            "gold_sql": "SELECT occupation, AVG(age) AS avg_age, AVG(hours_per_week) AS avg_hours FROM adult_income WHERE occupation <> '?' GROUP BY occupation ORDER BY avg_age DESC LIMIT 15;"
        },
        {
            "question": "What is the distribution of people by marital status and income?",
            "gold_sql": "SELECT marital_status, income, COUNT(*) AS cnt FROM adult_income GROUP BY marital_status, income ORDER BY marital_status, income;"
        },
        {
            "question": "For each relationship type, what percentage of people earn more than 50K?",
            "gold_sql": "SELECT relationship, 100.0 * AVG(CASE WHEN income = '>50K' THEN 1.0 ELSE 0.0 END) AS pct_high_income FROM adult_income GROUP BY relationship ORDER BY pct_high_income DESC;"
        },
        {
            "question": "Which workclass has the highest percentage of people with capital_gain > 0?",
            "gold_sql": "SELECT workclass, 100.0 * AVG(CASE WHEN capital_gain > 0 THEN 1.0 ELSE 0.0 END) AS pct_with_gain FROM adult_income GROUP BY workclass ORDER BY pct_with_gain DESC LIMIT 1;"
        },
        {
            "question": "For each education level, show the average capital gain and capital loss.",
            "gold_sql": "SELECT education, AVG(capital_gain) AS avg_capital_gain, AVG(capital_loss) AS avg_capital_loss FROM adult_income GROUP BY education ORDER BY avg_capital_gain DESC;"
        },
        {
            "question": "What is the average hours per week by age group (under 30, 30-50, over 50)?",
            "gold_sql": "SELECT CASE WHEN age < 30 THEN 'Under 30' WHEN age <= 50 THEN '30-50' ELSE 'Over 50' END AS age_group, AVG(hours_per_week) AS avg_hours FROM adult_income GROUP BY age_group ORDER BY age_group;"
        },
        {
            "question": "For each race, show the most common education level and its count.",
            "gold_sql": "WITH edu AS (SELECT race, education, COUNT(*) AS cnt FROM adult_income GROUP BY race, education), ranked AS (SELECT race, education, cnt, ROW_NUMBER() OVER (PARTITION BY race ORDER BY cnt DESC, education ASC) AS rn FROM edu) SELECT race, education AS top_education, cnt FROM ranked WHERE rn = 1 ORDER BY cnt DESC;"
        },
        {
            "question": "Which occupation has the highest average hours per week among people earning more than 50K (ignore '?', require at least 15 people)?",
            "gold_sql": "SELECT occupation, AVG(hours_per_week) AS avg_hours FROM adult_income WHERE income = '>50K' AND occupation <> '?' GROUP BY occupation HAVING COUNT(*) >= 15 ORDER BY avg_hours DESC LIMIT 1;"
        },
        {
            "question": "For each sex and marital status, what is the average educational_num?",
            "gold_sql": "SELECT sex, marital_status, AVG(educational_num) AS avg_educational_num FROM adult_income GROUP BY sex, marital_status ORDER BY sex, avg_educational_num DESC;"
        },
        {
            "question": "What is the percentage of people working more than 45 hours per week by income class?",
            "gold_sql": "SELECT income, 100.0 * AVG(CASE WHEN hours_per_week > 45 THEN 1.0 ELSE 0.0 END) AS pct_over_45hrs FROM adult_income GROUP BY income ORDER BY income;"
        },
        {
            "question": "For each native country, show the average age by income class (require at least 25 people per country).",
            "gold_sql": "SELECT native_country, income, AVG(age) AS avg_age FROM adult_income GROUP BY native_country, income HAVING COUNT(*) >= 25 ORDER BY native_country, income;"
        },
        {
            "question": "Which workclass has the highest average age?",
            "gold_sql": "SELECT workclass, AVG(age) AS avg_age FROM adult_income GROUP BY workclass ORDER BY avg_age DESC LIMIT 1;"
        },
        {
            "question": "For each relationship type, show the count and average hours per week.",
            "gold_sql": "SELECT relationship, COUNT(*) AS cnt, AVG(hours_per_week) AS avg_hours FROM adult_income GROUP BY relationship ORDER BY cnt DESC;"
        },
        {
            "question": "What is the average capital gain for people with different education levels (top 10 by average gain)?",
            "gold_sql": "SELECT education, AVG(capital_gain) AS avg_capital_gain FROM adult_income GROUP BY education ORDER BY avg_capital_gain DESC LIMIT 10;"
        },
        {
            "question": "For each sex, what is the most common occupation (ignore '?')?",
            "gold_sql": "WITH occ AS (SELECT sex, occupation, COUNT(*) AS cnt FROM adult_income WHERE occupation <> '?' GROUP BY sex, occupation), ranked AS (SELECT sex, occupation, cnt, ROW_NUMBER() OVER (PARTITION BY sex ORDER BY cnt DESC, occupation ASC) AS rn FROM occ) SELECT sex, occupation AS top_occupation, cnt FROM ranked WHERE rn = 1 ORDER BY cnt DESC;"
        },
        {
            "question": "Which education level has the highest average hours per week among people earning more than 50K?",
            "gold_sql": "SELECT education, AVG(hours_per_week) AS avg_hours FROM adult_income WHERE income = '>50K' GROUP BY education ORDER BY avg_hours DESC LIMIT 1;"
        },
        {
            "question": "For each race, show the percentage of people in each income bracket.",
            "gold_sql": "SELECT race, income, 100.0 * COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY race) AS pct FROM adult_income GROUP BY race, income ORDER BY race, income;"
        },
        {
            "question": "What is the average age by workclass and sex?",
            "gold_sql": "SELECT workclass, sex, AVG(age) AS avg_age FROM adult_income GROUP BY workclass, sex ORDER BY workclass, sex;"
        },
        {
            "question": "For each occupation, show the average educational_num (ignore '?', top 10 by average education).",
            "gold_sql": "SELECT occupation, AVG(educational_num) AS avg_educational_num FROM adult_income WHERE occupation <> '?' GROUP BY occupation ORDER BY avg_educational_num DESC LIMIT 10;"
        },
        {
            "question": "Which marital status has the highest percentage of people with capital_loss > 0?",
            "gold_sql": "SELECT marital_status, 100.0 * AVG(CASE WHEN capital_loss > 0 THEN 1.0 ELSE 0.0 END) AS pct_with_loss FROM adult_income GROUP BY marital_status ORDER BY pct_with_loss DESC LIMIT 1;"
        },
        {
            "question": "For each native country, show the average hours per week (require at least 30 people, top 15).",
            "gold_sql": "SELECT native_country, AVG(hours_per_week) AS avg_hours FROM adult_income GROUP BY native_country HAVING COUNT(*) >= 30 ORDER BY avg_hours DESC LIMIT 15;"
        },
        {
            "question": "What is the distribution of people by relationship type and income?",
            "gold_sql": "SELECT relationship, income, COUNT(*) AS cnt FROM adult_income GROUP BY relationship, income ORDER BY relationship, income;"
        },
        {
            "question": "For each education level, show the most common workclass and its count.",
            "gold_sql": "WITH wc AS (SELECT education, workclass, COUNT(*) AS cnt FROM adult_income GROUP BY education, workclass), ranked AS (SELECT education, workclass, cnt, ROW_NUMBER() OVER (PARTITION BY education ORDER BY cnt DESC, workclass ASC) AS rn FROM wc) SELECT education, workclass AS top_workclass, cnt FROM ranked WHERE rn = 1 ORDER BY cnt DESC;"
        },
        {
            "question": "Which race has the highest average capital gain?",
            "gold_sql": "SELECT race, AVG(capital_gain) AS avg_capital_gain FROM adult_income GROUP BY race ORDER BY avg_capital_gain DESC LIMIT 1;"
        },
        {
            "question": "For each sex, what is the average age and average educational_num?",
            "gold_sql": "SELECT sex, AVG(age) AS avg_age, AVG(educational_num) AS avg_educational_num FROM adult_income GROUP BY sex ORDER BY sex;"
        },
        {
            "question": "What is the percentage of people working less than 30 hours per week by income class?",
            "gold_sql": "SELECT income, 100.0 * AVG(CASE WHEN hours_per_week < 30 THEN 1.0 ELSE 0.0 END) AS pct_under_30hrs FROM adult_income GROUP BY income ORDER BY income;"
        },
        {
            "question": "For each occupation, show the average age (ignore '?', require at least 20 people, top 10).",
            "gold_sql": "SELECT occupation, AVG(age) AS avg_age FROM adult_income WHERE occupation <> '?' GROUP BY occupation HAVING COUNT(*) >= 20 ORDER BY avg_age DESC LIMIT 10;"
        }
    ]

NUM_TEST_QUESTIONS = len(TEST_CASES)
print(f"Loaded {NUM_TEST_QUESTIONS} test questions.")


Loaded 53 test questions.


In [14]:
# === Cell 3: MySQL helpers ===

def get_mysql_conn():
    return pymysql.connect(
        host=MYSQL_HOST,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DB,
        cursorclass=pymysql.cursors.DictCursor
    )

def get_schema_snippet():
    """Get a simple schema string like: adult_income(col1, col2, ...)."""
    conn = get_mysql_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(f"DESCRIBE {TABLE_NAME}")
            rows = cur.fetchall()
    finally:
        conn.close()

    schema = f"{TABLE_NAME}(" + ", ".join([row["Field"] for row in rows]) + ")"
    return schema

schema_snippet = get_schema_snippet()
print("Schema snippet:", schema_snippet)


Schema snippet: adult_income(age, workclass, fnlwgt, education, educational_num, marital_status, occupation, relationship, race, sex, capital_gain, capital_loss, hours_per_week, native_country, income)


In [15]:
# === Cell 4 (fixed): Model call & SQL execution with semantic support ===
import time

def call_model(prompt: str, retries=3, backoff=2) -> str:
    """
    Robust model call with retry, error handling, malformed response fallback.
    Prevents execution from stopping mid-test.
    """
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0
    }

    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(MODEL_ENDPOINT, json=payload, timeout=60)

            # HTTP or server errors
            if resp.status_code != 200:
                print(f"⚠️ Model returned {resp.status_code}: {resp.text}")
                raise RuntimeError(f"Bad response ({resp.status_code})")

            data = resp.json()

            # Validate structure
            if "choices" not in data or not data["choices"]:
                print("⚠️ Model returned empty choices, retrying...")
                raise ValueError("Empty choices response")

            msg = data["choices"][0].get("message")
            if not msg or "content" not in msg:
                print("⚠️ Malformed response:", data)
                raise ValueError("Missing content")

            return msg["content"].strip()

        except Exception as e:
            print(f"⚠️ Model call failed (attempt {attempt}/{retries}): {e}")
            time.sleep(backoff)

    # Guaranteed return to keep loop alive
    print("❌ Model failed after retries — returning fallback.")
    return "ERROR_FAILED_TO_GENERATE"


def run_sql(sql: str):
    """
    Run SQL against MySQL and return (success, rows, error_message).
    Now safer — prevents cursor/close failures from stalling evaluation.
    """
    sql_clean = sql.strip().rstrip(";") + ";"

    conn = get_mysql_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(sql_clean)
            rows = cur.fetchall()
        try:
            conn.close()
        except:
            pass
        return True, rows, None

    except Exception as e:
        try:
            conn.close()
        except:
            pass
        return False, None, str(e)


def normalize_rows(rows, float_decimals=4):
    """
    Normalizes rows so ordering, row order, and float precision differences
    don't affect correctness — now protects against None/empty.
    """
    if not rows:
        return []

    norm = []
    for row in rows:
        vals = []
        for v in row.values():
            if isinstance(v, float):
                v = round(v, float_decimals)
            vals.append(v)

        vals_as_str = sorted([repr(v) for v in vals])
        norm.append(tuple(vals_as_str))

    return sorted(norm)


In [16]:
# === Cell 5: Prompt-building for multiple techniques ===

def build_prompt_for_technique(
    technique: str,
    schema: str,
    question: str,
    examples,
    model_name: str,
) -> str:
    """
    Dispatch to the appropriate prompt builder based on technique.
    examples: list of {"question": ..., "sql": ...} for few-shot types.
    """
    technique = technique.lower()

    if technique == "few_shot":
        return build_few_shot_prompt(
            schema=schema,
            question=question,
            examples=examples,
            model_name=model_name,
        )

    elif technique == "cot":
        # Chain-of-Thought: ask the model to reason, but your prompts.py
        # already wraps it in the training-style Alpaca format.
        return build_cot_prompt(
            schema=schema,
            question=question,
            model_name=model_name,
        )

    elif technique == "ltm":
        # Least-to-Most style: again using your canned builder
        return build_ltm_prompt(
            schema=schema,
            question=question,
            model_name=model_name,
        )

    elif technique == "eg":
        # Execution-guided style prompt (one-shot, no looped execution here)
        return build_eg_prompt(
            schema=schema,
            question=question,
            model_name=model_name,
        )

    else:
        raise ValueError(f"Unknown prompting technique: {technique}")


In [17]:
# === Cell 6 (fully fixed): Evaluate prompting techniques safely ===

def evaluate_technique(technique: str, save_json: bool = True, refine_attempts=2):
    """
    Run all TEST_CASES using a prompting technique.
    Includes retry + refinement to avoid stalls and bad SQL output.
    Returns (results_list, exec_accuracy, semantic_accuracy)
    """

    print("\n" + "=" * 60)
    print(f"Evaluating technique: {technique}")
    print("=" * 60)

    schema = schema_snippet
    results = []

    # few-shot examples: first 3
    few_shot_examples = [
        {"question": t["question"], "sql": t["gold_sql"]}
        for t in TEST_CASES[:3]
    ]

    for i, case in enumerate(TEST_CASES, 1):
        question = case["question"]
        gold_sql = case["gold_sql"]

        print(f"\n[{i}/{NUM_TEST_QUESTIONS}] {question}")

        # --------------- build initial prompt ---------------
        prompt = build_prompt_for_technique(
            technique=technique,
            schema=schema,
            question=question,
            examples=few_shot_examples,
            model_name=MODEL_NAME,
        )

        # --------------- generate SQL ---------------
        model_sql = call_model(prompt)
        print("Model SQL:", model_sql)

        # Defensive cleanup
        if not model_sql or "ERROR" in model_sql or len(model_sql) < 5:
            print("⚠️ Bad initial SQL, forcing refine")
            model_sql = None

        # --------------- Execute gold SQL ---------------
        gold_exec_success, gold_rows, gold_error = run_sql(gold_sql)
        if not gold_exec_success:
            print("⚠️ Unexpected failure running gold SQL:", gold_error)

        # --------------- Try executing model SQL ---------------
        model_exec_success = False
        model_rows = None
        exec_error = None

        attempt_sql = model_sql

        for attempt in range(refine_attempts + 1):
            if attempt_sql:
                success, rows, err = run_sql(attempt_sql)

                if success:
                    model_exec_success = True
                    model_rows = rows
                    break

                # store last error
                exec_error = err
                print(f"🚨 Execution failed (attempt {attempt+1}): {err}")

            # REFINE pass
            refine_prompt = build_refine_prompt(
                schema=schema,
                question=question,
                previous_sql=(attempt_sql or "NULL"),
                error_msg=(exec_error or "NULL"),
                model_name=MODEL_NAME,
            )
            print("🔁 Refining SQL...")
            attempt_sql = call_model(refine_prompt)

        # --------------- semantic match ---------------
        semantic_success = False
        if gold_exec_success and model_exec_success:
            gold_norm = normalize_rows(gold_rows)
            model_norm = normalize_rows(model_rows)
            semantic_success = (gold_norm == model_norm)

        # --------------- store result ---------------
        results.append({
            "technique": technique,
            "question": question,
            "model_sql": attempt_sql,
            "gold_sql": gold_sql,
            "exec_success": model_exec_success,
            "semantic_success": semantic_success,
            "exec_error": exec_error,
            "rows": len(model_rows) if model_rows else 0,
        })

        print(
            f"Exec: {model_exec_success} | "
            f"Semantic: {semantic_success} | "
            f"Rows: {len(model_rows) if model_rows else 0}"
        )

    # --------------- metrics ---------------
    total = len(results)
    exec_successes = sum(1 for r in results if r["exec_success"])
    sem_successes = sum(1 for r in results if r["semantic_success"])

    exec_acc = (exec_successes / total) * 100 if total else 0
    sem_acc = (sem_successes / total) * 100 if total else 0

    print("\n" + "-" * 60)
    print(f"Technique: {technique}")
    print(f"Total Tests: {total}")
    print(f"Execution Accuracy: {exec_acc:.2f}% ({exec_successes}/{total})")
    print(f"Semantic Accuracy:  {sem_acc:.2f}% ({sem_successes}/{total})")
    print("-" * 60)

    # Save detailed JSON
    if save_json:
        out_path = SCRIPT_DIR / f"adult_model_test_results_{technique}.json"
        with open(out_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"Saved: {out_path}")

    return results, exec_acc, sem_acc


In [ ]:
# === Cell 7 (updated): Run all techniques & collect exec + semantic accuracy ===

all_results = {}
accuracy_records = []

for tech in PROMPTING_TECHNIQUES:
    res, exec_acc, sem_acc = evaluate_technique(technique=tech)
    all_results[tech] = res
    accuracy_records.append({
        "technique": tech,
        "exec_accuracy": exec_acc,
        "semantic_accuracy": sem_acc,
    })

acc_df = pd.DataFrame(accuracy_records)
display(acc_df)



Evaluating technique: few_shot

[1/53] What is the total number of people in each workclass?
Model SQL: SELECT workclass ,  count(*) FROM adult_income GROUP BY workclass;
Exec: True | Semantic: True | Rows: 9

[2/53] For each race, show the average age and average hours per week.
Model SQL: SELECT race ,  AVG(age) ,  AVG(hours_per_week) FROM adult_income GROUP BY race ORDER BY AVG(age) DESC;
Exec: True | Semantic: True | Rows: 5

[3/53] Which education level has the highest percentage of people earning more than 50K?
Model SQL: SELECT education ,  100.0 * AVG( CASE WHEN income  =  ">50K" THEN 1.0 ELSE 0.0 END ) AS pct_high_income FROM adult_income GROUP BY education ORDER BY pct_high_income DESC LIMIT 1;
Exec: True | Semantic: True | Rows: 1

[4/53] What is the average capital gain for people who work more than 50 hours per week?
Model SQL: SELECT avg(capital_gain) FROM adult_income WHERE hours_per_week  >  50;
Exec: True | Semantic: True | Rows: 1

[5/53] For each marital status, sho

In [ ]:
# === Cell 8 (updated): Plot both execution & semantic accuracy ===

plt.figure(figsize=(7, 4))

x = range(len(acc_df))
width = 0.35

plt.bar([i - width/2 for i in x], acc_df["exec_accuracy"], width, label="Execution")
plt.bar([i + width/2 for i in x], acc_df["semantic_accuracy"], width, label="Semantic")

plt.xticks(x, acc_df["technique"])
plt.ylabel("Accuracy (%)")
plt.title("Execution vs Semantic Accuracy by Prompting Technique")
plt.ylim(0, 100)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()


In [ ]:
bad_cases = []
for tech in PROMPTING_TECHNIQUES:
    for r in all_results[tech]:
        if r["exec_success"] and not r["semantic_success"]:
            bad_cases.append(r)
            
print("Total bad cases:", len(bad_cases))

for r in bad_cases[:5]:  # inspect a few
    print("\nQ:", r["question"])
    print("MODEL SQL:", r["model_sql"])
    print("GOLD SQL :", r["gold_sql"])
    print("MODEL ROWS:", r.get("rows"))


